In [21]:
# ============================================================
# 🚗 PAKWHEELS USED CARS — KAGGLE-READY DATASET PIPELINE
# ============================================================
#
# Features:
#   ✔ Pages 1-495
#   ✔ JSON-LD extraction
#   ✔ Retry failed requests
#   ✔ Handles 403 / 429 safely
#   ✔ Checkpoint every 10 pages
#   ✔ Resume capability
#   ✔ Duplicate removal
#   ✔ PKR price cleaning
#   ✔ Mileage -> numeric km
#   ✔ Engine -> numeric cc
#   ✔ Vehicle age
#   ✔ Price in millions
#   ✔ Final Kaggle-ready CSV
#
# ============================================================

import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import numpy as np
import re
import time
import os
from datetime import datetime


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_URL = "https://www.pakwheels.com/used-cars/search/-/featured_1/"

START_PAGE = 1
END_PAGE = 495

REQUEST_DELAY = 2.0
CHECKPOINT_EVERY = 10

RAW_FILE = "pakwheels_raw_checkpoint.csv"
CLEAN_FILE = "pakwheels_used_cars_clean.csv"
FAILED_FILE = "pakwheels_failed_pages.csv"


# ============================================================
# 2. HTTP SESSION
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)


# ============================================================
# 3. JSON-LD EXTRACTION
# ============================================================

def extract_cars(html, page_number, source_url):

    soup = BeautifulSoup(html, "html.parser")

    cars = []

    scripts = soup.find_all(
        "script",
        type="application/ld+json"
    )

    for script in scripts:

        try:
            raw_data = script.string or script.get_text()

            if not raw_data.strip():
                continue

            data = json.loads(raw_data)

        except Exception:
            continue

        # ----------------------------------------------------
        # JSON-LD can be dict OR list
        # ----------------------------------------------------

        if isinstance(data, dict):
            items = [data]

        elif isinstance(data, list):
            items = data

        else:
            continue

        # ----------------------------------------------------
        # PROCESS PRODUCTS
        # ----------------------------------------------------

        for item in items:

            if not isinstance(item, dict):
                continue

            item_type = item.get("@type")

            # @type can be:
            # "Product"
            # ["Product"]
            if isinstance(item_type, list):
                is_product = "Product" in item_type
            else:
                is_product = item_type == "Product"

            if not is_product:
                continue


            # =================================================
            # BRAND
            # =================================================

            brand = item.get("brand")

            if isinstance(brand, dict):
                brand = brand.get("name")


            # =================================================
            # BASIC INFO
            # =================================================

            name = item.get("name")

            description = item.get("description")

            year = item.get("modelDate")

            fuel_type = item.get("fuelType")

            transmission = item.get(
                "vehicleTransmission"
            )


            # =================================================
            # ENGINE
            # =================================================

            engine = item.get(
                "vehicleEngine"
            )

            if isinstance(engine, dict):
                engine = engine.get(
                    "engineDisplacement"
                )


            # =================================================
            # MILEAGE
            # =================================================

            mileage = item.get(
                "mileageFromOdometer"
            )

            if isinstance(mileage, dict):

                mileage = mileage.get(
                    "value"
                )


            # =================================================
            # IMAGE
            # =================================================

            image = item.get("image")

            if isinstance(image, list):

                image = (
                    image[0]
                    if len(image) > 0
                    else None
                )


            # =================================================
            # PRICE
            # =================================================

            price = None

            currency = None

            offers = item.get("offers")

            if isinstance(offers, dict):

                price = offers.get("price")

                currency = offers.get(
                    "priceCurrency"
                )


            # =================================================
            # RECORD
            # =================================================

            cars.append({

                "brand": brand,

                "name": name,

                "year": year,

                "fuel_type": fuel_type,

                "transmission": transmission,

                "engine": engine,

                "mileage": mileage,

                "price": price,

                "currency": currency,

                "image": image,

                "description": description,

                "page": page_number,

                "source_url": source_url,

                "scraped_date":
                    datetime.now().strftime(
                        "%Y-%m-%d"
                    )
            })

    return cars


# ============================================================
# 4. NUMERIC CLEANING FUNCTIONS
# ============================================================

def clean_price(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    # Remove PKR, commas, spaces, etc.
    value = re.sub(
        r"[^0-9.]",
        "",
        value
    )

    if value == "":
        return np.nan

    try:
        return float(value)

    except:
        return np.nan


# ------------------------------------------------------------

def clean_mileage(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    # Example:
    # "32,000 km"
    # "50,500 km"

    numbers = re.findall(
        r"[\d,]+",
        value
    )

    if not numbers:
        return np.nan

    number = numbers[0].replace(
        ",",
        ""
    )

    try:
        return float(number)

    except:
        return np.nan


# ------------------------------------------------------------

def clean_engine(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    # Example:
    # 1300cc
    # 1000cc
    # 1800 cc

    numbers = re.findall(
        r"\d+",
        value
    )

    if not numbers:
        return np.nan

    try:
        return float(numbers[0])

    except:
        return np.nan


# ============================================================
# 5. SCRAPE WITH CHECKPOINTS
# ============================================================

all_cars = []

failed_pages = []


# ------------------------------------------------------------
# RESUME FROM EXISTING CHECKPOINT
# ------------------------------------------------------------

if os.path.exists(RAW_FILE):

    print("Existing checkpoint found.")

    old_df = pd.read_csv(
        RAW_FILE
    )

    all_cars = old_df.to_dict(
        "records"
    )

    if len(old_df) > 0:

        completed_pages = (
            pd.to_numeric(
                old_df["page"],
                errors="coerce"
            )
            .dropna()
            .astype(int)
            .max()
        )

        START_PAGE = completed_pages + 1

        print(
            f"Resuming from page {START_PAGE}"
        )

else:

    print(
        "No checkpoint found."
    )

    print(
        "Starting from page 1."
    )


# ============================================================
# SCRAPING LOOP
# ============================================================

for page in range(
    START_PAGE,
    END_PAGE + 1
):

    # --------------------------------------------------------
    # URL
    # --------------------------------------------------------

    if page == 1:

        url = BASE_URL

    else:

        url = (
            f"{BASE_URL}"
            f"?page={page}"
        )


    print("\n" + "=" * 70)

    print(
        f"PAGE {page}/{END_PAGE}"
    )

    print(url)


    # --------------------------------------------------------
    # REQUEST
    # --------------------------------------------------------

    success = False

    for attempt in range(1, 4):

        try:

            response = session.get(
                url,
                timeout=30
            )

            print(
                f"Attempt {attempt} | "
                f"Status: {response.status_code}"
            )


            # ------------------------------------------------
            # RATE LIMIT / ACCESS RESPONSE
            # ------------------------------------------------

            if response.status_code in [403, 429]:

                print(
                    "Access/rate-limit response."
                )

                print(
                    "Stopping safely."
                )

                failed_pages.append({

                    "page": page,

                    "url": url,

                    "status":
                        response.status_code,

                    "reason":
                        "Access/rate limit"
                })

                success = False

                break


            # ------------------------------------------------
            # OTHER FAILED STATUS
            # ------------------------------------------------

            if response.status_code != 200:

                print(
                    "Request failed."
                )

                time.sleep(
                    5 * attempt
                )

                continue


            # ------------------------------------------------
            # EXTRACT
            # ------------------------------------------------

            cars = extract_cars(
                response.text,
                page,
                url
            )

            print(
                f"Cars extracted: {len(cars)}"
            )

            all_cars.extend(
                cars
            )

            success = True

            break


        except requests.RequestException as e:

            print(
                f"Request error: {e}"
            )

            time.sleep(
                5 * attempt
            )


    # --------------------------------------------------------
    # FAILED PAGE
    # --------------------------------------------------------

    if not success:

        if not any(
            x["page"] == page
            for x in failed_pages
        ):

            failed_pages.append({

                "page": page,

                "url": url,

                "status": "ERROR",

                "reason":
                    "Request failed after retries"
            })


    # --------------------------------------------------------
    # CHECKPOINT EVERY 10 PAGES
    # --------------------------------------------------------

    if (
        page % CHECKPOINT_EVERY == 0
        or page == END_PAGE
    ):

        checkpoint_df = pd.DataFrame(
            all_cars
        )

        checkpoint_df.to_csv(
            RAW_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "\n💾 CHECKPOINT SAVED"
        )

        print(
            f"Total raw records: "
            f"{len(checkpoint_df)}"
        )


        if failed_pages:

            pd.DataFrame(
                failed_pages
            ).to_csv(
                FAILED_FILE,
                index=False,
                encoding="utf-8-sig"
            )


    # --------------------------------------------------------
    # DELAY
    # --------------------------------------------------------

    time.sleep(
        REQUEST_DELAY
    )


# ============================================================
# 6. RAW DATAFRAME
# ============================================================

raw_df = pd.DataFrame(
    all_cars
)

print("\n")
print("=" * 70)
print("RAW DATA COMPLETE")
print("=" * 70)

print(
    "Raw rows:",
    len(raw_df)
)


# ============================================================
# 7. REMOVE DUPLICATES
# ============================================================

if len(raw_df) > 0:

    before = len(raw_df)

    # --------------------------------------------------------
    # First remove exact duplicates
    # --------------------------------------------------------

    raw_df = raw_df.drop_duplicates(
        keep="first"
    )


    # --------------------------------------------------------
    # Then remove repeated listings
    # --------------------------------------------------------

    duplicate_columns = [
        "name",
        "price",
        "image"
    ]

    available_columns = [
        c for c in duplicate_columns
        if c in raw_df.columns
    ]

    if available_columns:

        raw_df = raw_df.drop_duplicates(
            subset=available_columns,
            keep="first"
        )


    raw_df = raw_df.reset_index(
        drop=True
    )

    print(
        "Duplicates removed:",
        before - len(raw_df)
    )

    print(
        "Rows after deduplication:",
        len(raw_df)
    )


# ============================================================
# 8. CLEAN PRICE
# ============================================================

raw_df["price"] = raw_df[
    "price"
].apply(
    clean_price
)


# ============================================================
# 9. CLEAN MILEAGE
# ============================================================

raw_df["mileage_km"] = raw_df[
    "mileage"
].apply(
    clean_mileage
)


# ============================================================
# 10. CLEAN ENGINE
# ============================================================

raw_df["engine_cc"] = raw_df[
    "engine"
].apply(
    clean_engine
)


# ============================================================
# 11. CLEAN YEAR
# ============================================================

raw_df["year"] = pd.to_numeric(
    raw_df["year"],
    errors="coerce"
)


# ============================================================
# 12. VEHICLE AGE
# ============================================================

CURRENT_YEAR = datetime.now().year

raw_df["vehicle_age"] = (
    CURRENT_YEAR
    - raw_df["year"]
)


# ============================================================
# 13. PRICE IN MILLIONS
# ============================================================

raw_df["price_million"] = (
    raw_df["price"]
    / 1_000_000
)


# ============================================================
# 14. VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("DATA VALIDATION")
print("=" * 70)


print("\nMissing values:")

print(
    raw_df.isnull().sum()
)


# ============================================================
# 15. REMOVE INVALID RECORDS
# ============================================================

# Year should be realistic
raw_df.loc[
    ~raw_df["year"].between(
        1950,
        CURRENT_YEAR + 1
    ),
    "year"
] = np.nan


# Price should be positive
raw_df.loc[
    raw_df["price"] <= 0,
    "price"
] = np.nan


# Mileage cannot be negative
raw_df.loc[
    raw_df["mileage_km"] < 0,
    "mileage_km"
] = np.nan


# Engine cannot be negative
raw_df.loc[
    raw_df["engine_cc"] < 0,
    "engine_cc"
] = np.nan


# Recalculate derived columns
raw_df["vehicle_age"] = (
    CURRENT_YEAR
    - raw_df["year"]
)

raw_df["price_million"] = (
    raw_df["price"]
    / 1_000_000
)


# ============================================================
# 16. REMOVE COMPLETELY USELESS RECORDS
# ============================================================

raw_df = raw_df.dropna(
    subset=[
        "name",
        "price"
    ]
)


raw_df = raw_df.reset_index(
    drop=True
)


# ============================================================
# 17. DATA TYPES
# ============================================================

raw_df["year"] = (
    raw_df["year"]
    .astype("Int64")
)

raw_df["price"] = (
    raw_df["price"]
    .round(0)
    .astype("Int64")
)

raw_df["mileage_km"] = (
    raw_df["mileage_km"]
    .round(0)
    .astype("Int64")
)

raw_df["engine_cc"] = (
    raw_df["engine_cc"]
    .round(0)
    .astype("Int64")
)

raw_df["vehicle_age"] = (
    raw_df["vehicle_age"]
    .round(0)
    .astype("Int64")
)

raw_df["price_million"] = (
    raw_df["price_million"]
    .round(3)
)


# ============================================================
# 18. REORDER COLUMNS
# ============================================================

final_columns = [

    "brand",
    "name",
    "year",

    "vehicle_age",

    "fuel_type",
    "transmission",

    "engine",
    "engine_cc",

    "mileage",
    "mileage_km",

    "price",
    "price_million",

    "currency",

    "image",
    "description",

    "page",
    "source_url",

    "scraped_date"
]


# Only keep columns that exist
final_columns = [
    c for c in final_columns
    if c in raw_df.columns
]


raw_df = raw_df[
    final_columns
]


# ============================================================
# 19. FINAL DUPLICATE CHECK
# ============================================================

raw_df = raw_df.drop_duplicates(
    subset=[
        "name",
        "year",
        "price",
        "mileage_km",
        "image"
    ],
    keep="first"
)


raw_df = raw_df.reset_index(
    drop=True
)


# ============================================================
# 20. SAVE FINAL KAGGLE DATASET
# ============================================================

raw_df.to_csv(
    CLEAN_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print("\n")
print("=" * 70)
print("🚀 KAGGLE DATASET READY")
print("=" * 70)

print(
    f"Final rows: {len(raw_df):,}"
)

print(
    f"Final columns: {len(raw_df.columns)}"
)

print(
    f"CSV file: {CLEAN_FILE}"
)


# ============================================================
# 22. DATASET PREVIEW
# ============================================================

print("\nFirst 10 rows:")

display(
    raw_df.head(10)
)


# ============================================================
# 23. DATASET INFO
# ============================================================

print("\nDataset information:")

print(
    raw_df.info()
)


# ============================================================
# 24. STATISTICS
# ============================================================

print("\nNumerical statistics:")

display(
    raw_df[
        [
            "year",
            "vehicle_age",
            "engine_cc",
            "mileage_km",
            "price",
            "price_million"
        ]
    ].describe()
)


# ============================================================
# 25. TOP BRANDS
# ============================================================

print("\nTop 15 brands:")

display(
    raw_df["brand"]
    .value_counts()
    .head(15)
)


# ============================================================
# 26. TOP FUEL TYPES
# ============================================================

print("\nFuel types:")

display(
    raw_df["fuel_type"]
    .value_counts()
)


# ============================================================
# 27. TRANSMISSIONS
# ============================================================

print("\nTransmission types:")

display(
    raw_df["transmission"]
    .value_counts()
)


# ============================================================
# 28. FINAL MISSING VALUES
# ============================================================

print("\nFinal missing values:")

display(
    raw_df.isnull().sum()
)


print("\n" + "=" * 70)

print("DONE ✅")

print("=" * 70)


No checkpoint found.
Starting from page 1.

PAGE 1/495
https://www.pakwheels.com/used-cars/search/-/featured_1/
Attempt 1 | Status: 200
Cars extracted: 25

PAGE 2/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=2
Attempt 1 | Status: 200
Cars extracted: 37

PAGE 3/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=3
Attempt 1 | Status: 200
Cars extracted: 32

PAGE 4/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=4
Attempt 1 | Status: 200
Cars extracted: 37

PAGE 5/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=5
Attempt 1 | Status: 200
Cars extracted: 32

PAGE 6/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=6
Attempt 1 | Status: 200
Cars extracted: 37

PAGE 7/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=7
Attempt 1 | Status: 200
Cars extracted: 37

PAGE 8/495
https://www.pakwheels.com/used-cars/search/-/featured_1/?page=8
Attempt 1 | Status: 200
Cars extracted: 36

PAGE 9/495


,brand,name,year,vehicle_age,fuel_type,transmission,engine,engine_cc,mileage,mileage_km,price,price_million,currency,image,description,page,source_url,scraped_date
0,Honda,Honda Civic 2007 for sale in Bahawalpur,2007,19,Petrol,Manual,1800cc,1800,"130,000 km",130000,2200000,2.200,PKR,https://cache1.pakwheels.com/ad_pictures/1483/...,Honda Civic 2007 for sale in Bahawalpur,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
1,Honda,Honda Civic 2014 for sale in Bahawalpur,2014,12,Petrol,Automatic,1800cc,1800,"121,000 km",121000,3500000,3.500,PKR,https://cache4.pakwheels.com/ad_pictures/1522/...,Honda Civic 2014 for sale in Bahawalpur,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
2,Toyota,Toyota Corolla 2011 for sale in Faisalabad,2011,15,Petrol,Manual,1300cc,1300,"260,000 km",260000,2300000,2.300,PKR,https://cache4.pakwheels.com/ad_pictures/1522/...,Toyota Corolla 2011 for sale in Faisalabad,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
3,Toyota,Toyota Corolla 2011 for sale in Bahawalpur,2011,15,Petrol,Manual,1300cc,1300,"150,000 km",150000,3000000,3.000,PKR,https://cache2.pakwheels.com/ad_pictures/1470/...,Toyota Corolla 2011 for sale in Bahawalpur,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
4,Honda,Honda City 2010 for sale in Rawalpindi,2010,16,Petrol,Manual,1300cc,1300,"190,000 km",190000,2650000,2.650,PKR,https://cache3.pakwheels.com/ad_pictures/1519/...,Honda City 2010 for sale in Rawalpindi,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
5,Toyota,Toyota Raize 2020 for sale in Lahore,2020,6,Petrol,Automatic,1000cc,1000,"50,500 km",50500,5800000,5.800,PKR,https://cache2.pakwheels.com/ad_pictures/1510/...,Toyota Raize 2020 for sale in Lahore,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
6,Toyota,Toyota Passo 2021 for sale in Lahore,2021,5,Petrol,Automatic,1000cc,1000,"32,000 km",32000,3980000,3.980,PKR,https://cache4.pakwheels.com/ad_pictures/1509/...,Toyota Passo 2021 for sale in Lahore,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
7,Toyota,Toyota Corolla Axio 2007 for sale in Islamabad,2007,19,Petrol,Automatic,1500cc,1500,"327,000 km",327000,3500000,3.500,PKR,https://cache2.pakwheels.com/ad_pictures/1522/...,Toyota Corolla Axio 2007 for sale in Islamabad,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
8,Honda,Honda Civic 2021 for sale in Islamabad,2021,5,Petrol,Automatic,1800cc,1800,"48,016 km",48016,6650000,6.650,PKR,https://cache4.pakwheels.com/ad_pictures/1522/...,Honda Civic 2021 for sale in Islamabad,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25
9,Mitsubishi,Mitsubishi Lancer 2005 for sale in Islamabad,2005,21,Petrol,Manual,1300cc,1300,"158,455 km",158455,1745000,1.745,PKR,https://cache1.pakwheels.com/ad_pictures/1519/...,Mitsubishi Lancer 2005 for sale in Islamabad,1,https://www.pakwheels.com/used-cars/search/-/f...,2026-09-25



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11630 entries, 0 to 11629
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   brand          11630 non-null  object 
 1   name           11630 non-null  object 
 2   year           11630 non-null  Int64  
 3   vehicle_age    11630 non-null  Int64  
 4   fuel_type      11630 non-null  object 
 5   transmission   11630 non-null  object 
 6   engine         11630 non-null  object 
 7   engine_cc      11417 non-null  Int64  
 8   mileage        11630 non-null  object 
 9   mileage_km     11630 non-null  Int64  
 10  price          11630 non-null  Int64  
 11  price_million  11630 non-null  float64
 12  currency       11630 non-null  object 
 13  image          11592 non-null  object 
 14  description    11630 non-null  object 
 15  page           11630 non-null  int64  
 16  source_url     11630 non-null  object 
 17  scraped_date   11630 non-nul

,year,vehicle_age,engine_cc,mileage_km,price,price_million
count,11630.0,11630.0,11417.0,11630.0,11630.0,11630.000000
mean,2018.658727,7.341273,1783.482701,71739.601376,9333547.063715,9.333547
std,6.492317,6.492317,934.202455,59521.594455,11298293.822998,11.298294
min,1965.0,0.0,0.0,1.0,350000.0,0.350000
25%,2016.0,3.0,1300.0,27000.0,3900000.0,3.900000
50%,2021.0,5.0,1500.0,62000.0,5860000.0,5.860000
75%,2023.0,10.0,2000.0,100000.0,9800000.0,9.800000
max,2026.0,61.0,9999.0,910000.0,175000000.0,175.000000



Top 15 brands:


...


Transmission types:


transmission
Automatic    10307
Manual        1323
Name: count, dtype: int64


Final missing values:


brand              0
name               0
year               0
vehicle_age        0
fuel_type          0
transmission       0
engine             0
engine_cc        213
mileage            0
mileage_km         0
price              0
price_million      0
currency           0
image             38
description        0
page               0
source_url         0
scraped_date       0
dtype: int64


DONE ✅
